# dragon-farm: train a bundled run on this GPU

This notebook trains a run that `dragonfarm::dragon_bundle()` packaged in R. It works the same on
Google Colab, Kaggle, Lightning AI, and RunPod. Make sure the session has a GPU, run all cells, and
download the `dragonfarm-results-*.zip` the last cell produces. Back in R, `dragon_import()` puts
the trained adapter into the run directory.

- **Colab:** Runtime > Change runtime type > T4 GPU. The first cell asks you to upload the bundle.
- **Kaggle:** Session options > Accelerator = GPU, Internet = On. Add the bundle as a dataset (Input > Upload).
- **Lightning AI / RunPod:** upload the bundle next to this notebook in the file browser.

In [ ]:
# 1. Find the bundle (dragonfarm-<run id>.zip) that dragon_bundle() created.
import glob
import os
import sys


def find_bundle():
    patterns = [
        "dragonfarm-*.zip", "*/dragonfarm-*.zip",
        "/kaggle/input/**/dragonfarm-*.zip",
        "/content/dragonfarm-*.zip", "/content/drive/MyDrive/dragonfarm-*.zip",
        "/teamspace/**/dragonfarm-*.zip",
        "/workspace/dragonfarm-*.zip", "/workspace/**/dragonfarm-*.zip",
    ]
    hits = []
    for p in patterns:
        hits += glob.glob(p, recursive=True)
    hits = [h for h in hits if "results" not in os.path.basename(h)]
    return max(hits, key=os.path.getmtime) if hits else None


bundle = find_bundle()
if bundle is None:
    try:
        from google.colab import files  # Colab: ask for the upload

        print("Choose the dragonfarm-*.zip file that dragon_bundle() created.")
        uploaded = files.upload()
        bundle = next(iter(uploaded))
    except ImportError:
        raise SystemExit(
            "No dragonfarm-*.zip found. Upload the bundle next to this notebook "
            "(on Kaggle: add it as a dataset under Input) and run this cell again."
        )
print("bundle:", bundle)

In [ ]:
# 2. Unpack the bundle and install the libraries the trainer needs.
import pathlib
import shutil
import subprocess
import zipfile

WORK = pathlib.Path("dragonfarm_work").resolve()
if WORK.exists():
    shutil.rmtree(WORK)
zipfile.ZipFile(bundle).extractall(WORK)
RUN = WORK / "run"
assert (RUN / "config.json").exists(), "the zip does not look like a dragonfarm bundle"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(WORK / "requirements.txt")], check=True)

import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("WARNING: no GPU in this session. Colab: Runtime > Change runtime type. "
          "Kaggle: Session options > Accelerator. Training will be very slow on the CPU.")

In [ ]:
# 3. Optional: a Hugging Face token, needed only for gated models such as Gemma and Llama.
#    Store it as a secret named HF_TOKEN (Colab: key icon in the sidebar; Kaggle: Add-ons > Secrets).
if not os.environ.get("HF_TOKEN"):
    def from_colab():
        from google.colab import userdata
        return userdata.get("HF_TOKEN")

    def from_kaggle():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")

    for getter in (from_colab, from_kaggle):
        try:
            token = getter()
        except Exception:
            token = None
        if token:
            os.environ["HF_TOKEN"] = token
            break
print("HF token:", "set" if os.environ.get("HF_TOKEN") else "not set (fine for open models)")

In [ ]:
# 4. Train. This is the same trainer dragonfarm runs locally; progress prints as it goes.
import json

env = dict(os.environ, PYTHONPATH=str(WORK / "python"), PYTHONUNBUFFERED="1",
           TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1")
proc = subprocess.Popen(
    [sys.executable, "-m", "dragonfarm.train", "--run-dir", str(RUN)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

status = json.load(open(RUN / "status.json", encoding="utf-8"))
print("\nstate:", status.get("state"))
if status.get("error"):
    print("error:", status["error"])

In [ ]:
# 5. Pack the results for dragon_import() and hand them back.
out = subprocess.run(
    [sys.executable, "-m", "dragonfarm.pack_results", str(RUN), "--out", os.getcwd()],
    env=env, capture_output=True, text=True, check=True,
).stdout.strip()
print("results:", out)

try:
    from google.colab import files

    files.download(out)
except ImportError:
    print("Download this file from the file browser (on Kaggle it is under Output after you save the notebook).")

run_id = json.load(open(RUN / "config.json", encoding="utf-8"))["run_id"]
print(f'\nBack in R:\n  dragon_import(dragon_run("dragonfarm_runs/{run_id}"), "{os.path.basename(out)}")')